# Notebook 02: Extract Real Confidence Scores
# 
 Loads real NIH ChestX-ray14 images through pre-trained DenseNet-121
 and saves confidence scores + ground truth labels to data/scores.csv
#
 SSDI Connection: This creates the raw data for all statistical tests
 Units 1-7 all operate on the output of this notebook

In [5]:
import torch
import torchxrayvision as xrv
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import os
from PIL import Image

# ── Constants ────────────────────────────────────────────────────────────────
IMAGE_DIR   = "../data/images/images"   # where extracted PNGs live
CSV_PATH    = "../data/Data_Entry_2017_v2020.csv"
OUTPUT_CSV  = "../data/scores.csv"
PATHOLOGY   = "Effusion"
BATCH_SIZE  = 32
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Image folder: {IMAGE_DIR}")
print(f"Images available: {len(os.listdir(IMAGE_DIR))}")

Device: cpu
Image folder: ../data/images/images
Images available: 10000


In [6]:
# Load the NIH label CSV and match to available images
df_labels = pd.read_csv(CSV_PATH)

# Get list of images we actually have on disk
available_images = set(os.listdir(IMAGE_DIR))

# Filter labels to only rows where image exists on disk
df_labels = df_labels[df_labels["Image Index"].isin(available_images)].copy()

# Create binary ground truth: 1 if Effusion in Finding Labels, 0 otherwise
df_labels["y_true"] = df_labels["Finding Labels"].apply(
    lambda x: 1 if PATHOLOGY in str(x) else 0
)

print(f"Total matched images: {len(df_labels)}")
print(f"Effusion positive: {df_labels['y_true'].sum()} ({df_labels['y_true'].mean()*100:.2f}%)")
print(f"Effusion negative: {(df_labels['y_true']==0).sum()}")

Total matched images: 10000
Effusion positive: 895 (8.95%)
Effusion negative: 9105


In [7]:
# Load pre-trained DenseNet-121 with NIH weights
# eval() disables dropout and batch norm updates — we are auditing not training
model = xrv.models.DenseNet(weights="densenet121-res224-nih")
model.eval()
model = model.to(DEVICE)

# Find which output index is Effusion
PATHOLOGY_IDX = model.pathologies.index(PATHOLOGY)
print(f"Model loaded. {PATHOLOGY} is at output index: {PATHOLOGY_IDX}")
print(f"All pathologies: {model.pathologies}")

Model loaded. Effusion is at output index: 7
All pathologies: ['Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia', 'Pleural_Thickening', 'Cardiomegaly', 'Nodule', 'Mass', 'Hernia', '', '', '', '']


In [8]:
# Run inference on all matched images
# For each image: load → normalise → model → save probability + logit
# raw_logit: logit(prob) = log(p/(1-p)) — inverse sigmoid, needed for Temperature Scaling
# confidence_score: model output (sigmoid already applied internally)

all_scores  = []
all_logits  = []
all_labels  = []
all_ids     = []
errors      = 0

print(f"Running inference on {len(df_labels)} images...")
print("This will take 10-20 minutes on CPU. Progress shown every 500 images.")

for i, row in enumerate(df_labels.itertuples()):
    try:
        # Load image as grayscale
        img_path = os.path.join(IMAGE_DIR, row._1)
        img = Image.open(img_path).convert("L")  # L = grayscale
        img_array = np.array(img).astype(np.float32)

        # Normalise to [-1024, 1024] range as torchxrayvision expects
        img_array = xrv.datasets.normalize(img_array, maxval=255, reshape=True)

        # Resize to 224x224
        transform = transforms.Compose([
            xrv.datasets.XRayCenterCrop(),
            xrv.datasets.XRayResizer(224)
        ])
        img_tensor = transform(img_array)
        img_tensor = torch.from_numpy(img_tensor).unsqueeze(0).to(DEVICE)

        # Forward pass — model applies sigmoid internally, output is probability in [0,1]
        with torch.no_grad():
            output = model(img_tensor)

        # Capture probability directly (sigmoid already applied by model)
        prob = output[0, PATHOLOGY_IDX].item()

        # Reverse sigmoid to recover effective logit: logit(p) = log(p / (1-p))
        # Clip to avoid log(0) — any prob outside (1e-7, 1-1e-7) gets clipped
        p_clipped = float(np.clip(prob, 1e-7, 1 - 1e-7))
        raw_logit = float(np.log(p_clipped / (1 - p_clipped)))

        all_scores.append(prob)
        all_logits.append(raw_logit)
        all_labels.append(row.y_true)
        all_ids.append(row._1)

    except Exception as e:
        errors += 1
        continue

    if (i + 1) % 500 == 0:
        print(f"  Processed {i+1}/{len(df_labels)} images...")

print(f"\nInference complete. Errors skipped: {errors}")

Running inference on 10000 images...
This will take 10-20 minutes on CPU. Progress shown every 500 images.
  Processed 500/10000 images...
  Processed 1000/10000 images...
  Processed 1500/10000 images...
  Processed 2000/10000 images...
  Processed 2500/10000 images...
  Processed 3000/10000 images...
  Processed 3500/10000 images...
  Processed 4000/10000 images...
  Processed 4500/10000 images...
  Processed 5000/10000 images...
  Processed 5500/10000 images...
  Processed 6000/10000 images...
  Processed 6500/10000 images...
  Processed 7000/10000 images...
  Processed 7500/10000 images...
  Processed 8000/10000 images...
  Processed 8500/10000 images...
  Processed 9000/10000 images...
  Processed 9500/10000 images...
  Processed 10000/10000 images...

Inference complete. Errors skipped: 0


In [9]:
# Save to scores.csv — the master data file for all statistical analysis
scores_df = pd.DataFrame({
    "image_id":         all_ids,
    "ground_truth":     all_labels,
    "confidence_score": all_scores,
    "raw_logit":        all_logits,   # logit(confidence_score) — for Temperature Scaling
})

# Data validation before saving
assert scores_df["confidence_score"].between(0,1).all(), "Scores must be in [0,1]"
assert scores_df["ground_truth"].isin([0,1]).all(), "Labels must be 0 or 1"
assert scores_df.notna().all().all(), "No NaN values allowed"

scores_df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(scores_df):,} rows to {OUTPUT_CSV}")
print()
print("=== SCORES.CSV SUMMARY ===")
print(f"Total samples:      {len(scores_df):,}")
print(f"Effusion cases:    {scores_df['ground_truth'].sum():,} ({scores_df['ground_truth'].mean()*100:.2f}%)")
print(f"Negative cases:     {(scores_df['ground_truth']==0).sum():,}")
print(f"Mean confidence:    {scores_df['confidence_score'].mean():.4f}")
print(f"True prevalence:    {scores_df['ground_truth'].mean():.4f}")
print(f"Calibration gap:    {scores_df['confidence_score'].mean() - scores_df['ground_truth'].mean():+.4f}")
print(f"Score range:        [{scores_df['confidence_score'].min():.4f}, {scores_df['confidence_score'].max():.4f}]")
print(f"Logit range:        [{scores_df['raw_logit'].min():.4f}, {scores_df['raw_logit'].max():.4f}]")
print()
print("First 5 rows:")
print(scores_df.head().to_string(index=False))

Saved 10,000 rows to ../data/scores.csv

=== SCORES.CSV SUMMARY ===
Total samples:      10,000
Effusion cases:    895 (8.95%)
Negative cases:     9,105
Mean confidence:    0.2655
True prevalence:    0.0895
Calibration gap:    +0.1760
Score range:        [0.0001, 0.9782]
Logit range:        [-9.3177, 3.8038]

First 5 rows:
        image_id  ground_truth  confidence_score  raw_logit
00001336_000.png             0          0.061449  -2.726135
00001337_000.png             0          0.027735  -3.556947
00001338_000.png             0          0.026799  -3.592239
00001338_001.png             0          0.509365   0.037465
00001338_002.png             0          0.011469  -4.456574


# Extension: Save All 14 Disease Scores
#
 The model already computed all 14 pathology scores
 during inference. This cell saves them all to
 data/scores_all_diseases.csv for the multi-disease
 audit in notebook 09.
#
 data/scores.csv stays unchanged — Effusion only
 data/scores_all_diseases.csv — all 14 diseases

In [10]:
import pandas as pd
import numpy as np
import torchxrayvision as xrv
import torch
import torchvision.transforms as transforms
from PIL import Image
import os

# ── Constants ────────────────────────────────────────────────────────────────
IMAGE_DIR  = "../data/images/images"
CSV_PATH   = "../data/Data_Entry_2017_v2020.csv"
OUTPUT_CSV = "../data/scores_all_diseases.csv"
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# The 14 NIH pathologies we audit
# These match the NIH CSV Finding Labels column
NIH_PATHOLOGIES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia"
]

print("Loading model...")
model = xrv.models.DenseNet(weights="densenet121-res224-nih")
model.eval()
model = model.to(DEVICE)
print(f"Model pathologies: {model.pathologies}")
print()

# Find index of each NIH pathology in model output
# Model uses slightly different names so we map them
pathology_map = {}
for nih_name in NIH_PATHOLOGIES:
    for i, model_name in enumerate(model.pathologies):
        if nih_name.lower().replace("_", "") in model_name.lower().replace("_", ""):
            pathology_map[nih_name] = i
            break

print("Pathology index mapping:")
for name, idx in pathology_map.items():
    print(f"  {name:<20} \u2192 index {idx} ({model.pathologies[idx]})")
print()

# Load labels CSV
labels_df = pd.read_csv(CSV_PATH)
available  = set(os.listdir(IMAGE_DIR))
labels_df  = labels_df[labels_df["Image Index"].isin(available)].copy()
print(f"Matched images: {len(labels_df):,}")

# Create binary ground truth for each pathology
for path in NIH_PATHOLOGIES:
    # NIH CSV uses slightly different names
    csv_name = path.replace("_", " ")
    if path == "Pleural_Thickening":
        csv_name = "Pleural Thickening"
    labels_df[f"gt_{path}"] = labels_df["Finding Labels"].apply(
        lambda x, cn=csv_name: 1 if cn in str(x) else 0
    )

print("Ground truth positive rates:")
for path in NIH_PATHOLOGIES:
    rate = labels_df[f"gt_{path}"].mean()
    print(f"  {path:<25} {rate*100:.2f}%")
print()

# Run inference and collect all 14 scores
print("Running inference on all images...")
print("This will take 15-20 minutes on CPU...")

all_scores  = {path: [] for path in NIH_PATHOLOGIES}
all_gt      = {path: [] for path in NIH_PATHOLOGIES}
all_ids     = []
errors      = 0

for i, row in enumerate(labels_df.itertuples()):
    try:
        img_path  = os.path.join(IMAGE_DIR, row._1)
        img       = Image.open(img_path).convert("L")
        img_array = np.array(img).astype(np.float32)
        img_array = xrv.datasets.normalize(img_array, maxval=255, reshape=True)

        transform   = transforms.Compose([
            xrv.datasets.XRayCenterCrop(),
            xrv.datasets.XRayResizer(224)
        ])
        img_tensor  = transform(img_array)
        img_tensor  = torch.from_numpy(img_tensor).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(img_tensor)

        # Save score for each pathology
        # NOTE: model output is already probability (sigmoid applied internally)
        # Use .item() directly — do NOT wrap in torch.sigmoid() again
        for path in NIH_PATHOLOGIES:
            if path in pathology_map:
                idx   = pathology_map[path]
                prob  = output[0, idx].item()  # already a probability in [0,1]
                all_scores[path].append(prob)
                all_gt[path].append(
                    getattr(row, f"gt_{path}", 0)
                )

        all_ids.append(row._1)

    except Exception as e:
        errors += 1
        # Add placeholder for failed images to keep alignment
        for path in NIH_PATHOLOGIES:
            all_scores[path].append(0.5)
            all_gt[path].append(0)
        all_ids.append(getattr(row, "_1", "unknown"))
        continue

    if (i + 1) % 1000 == 0:
        print(f"  Processed {i+1}/{len(labels_df)} images...")

print(f"Inference complete. Errors: {errors}")
print()

# Build DataFrame
data = {"image_id": all_ids}

# Add confidence scores for each pathology
for path in NIH_PATHOLOGIES:
    data[f"score_{path}"] = all_scores[path]

# Add ground truth for each pathology
for path in NIH_PATHOLOGIES:
    data[f"gt_{path}"] = all_gt[path]

df_all = pd.DataFrame(data)

# Validate
assert len(df_all) == len(labels_df), "Row count mismatch"
assert df_all.notna().all().all(), "NaN values found"

df_all.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(df_all):,} rows to {OUTPUT_CSV}")
print(f"Columns: {len(df_all.columns)} total")
print()
print("=== ALL DISEASE SUMMARY ===")
print(f"{'Disease':<25} {'Prevalence':>12} {'Mean Score':>12} {'Gap':>10}")
print("-" * 62)
for path in NIH_PATHOLOGIES:
    gt_col    = f"gt_{path}"
    score_col = f"score_{path}"
    if gt_col in df_all.columns and score_col in df_all.columns:
        prevalence = df_all[gt_col].mean()
        mean_score = df_all[score_col].mean()
        gap        = mean_score - prevalence
        print(f"{path:<25} {prevalence*100:>11.2f}% "
              f"{mean_score:>12.4f} {gap:>+10.4f}")


Loading model...
Model pathologies: ['Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia', 'Pleural_Thickening', 'Cardiomegaly', 'Nodule', 'Mass', 'Hernia', '', '', '', '']

Pathology index mapping:
  Atelectasis          → index 0 (Atelectasis)
  Cardiomegaly         → index 10 (Cardiomegaly)
  Effusion             → index 7 (Effusion)
  Infiltration         → index 2 (Infiltration)
  Mass                 → index 12 (Mass)
  Nodule               → index 11 (Nodule)
  Pneumonia            → index 8 (Pneumonia)
  Pneumothorax         → index 3 (Pneumothorax)
  Consolidation        → index 1 (Consolidation)
  Edema                → index 4 (Edema)
  Emphysema            → index 5 (Emphysema)
  Fibrosis             → index 6 (Fibrosis)
  Pleural_Thickening   → index 9 (Pleural_Thickening)
  Hernia               → index 13 (Hernia)

Matched images: 10,000
Ground truth positive rates:
  Atelectasis               8.90%
  Ca